In [ ]:
import numpy as np
import pandas as pd

from sklearn import datasets
from sklearn.linear_model import Perceptron
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [ ]:
# @title
pd.options.mode.chained_assignment = None

# Carga do dataset do repositório do sklearn
iris = datasets.load_iris()

# Colocando no Pandas para filtrar
iris_df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
iris_df["target"] = iris.target

# Vamos manter apenas as duas primeiras classes: Iris Setosa e Iris Versicolour
vetores_classe_0 = iris_df[iris_df["target"] == 0]
vetores_classe_1 = iris_df[iris_df["target"] == 1]

# Removendo colunas, para deixar o problema bidimensional
remover = ['petal length (cm)', 'petal width (cm)']
vetores_classe_0.drop(columns = remover,inplace = True)
vetores_classe_1.drop(columns = remover,inplace = True)

# Ajustando classes para operar com o perceptron
vetores_classe_0["target"] = -1
vetores_classe_1["target"] = +1

# Colocando em um vetor numpy para facilitar
vetores = np.concatenate((vetores_classe_0.to_numpy(), vetores_classe_1.to_numpy()))

In [ ]:
# Use KFold paraf fazer um teste de 5 folds. Mostre a matriz de confusão final, e a acurácia média

X = vetores[:, [0, 1]] # Features (sepal length, sepal width)
y = vetores[:, 2]      # Target (-1, 1)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
classifier = Perceptron(eta0=0.1)

# Listas para armazenar os resultados de cada fold
todas_predicoes = []
todos_reais = []
acuracias = []

for train_index, test_index in kf.split(X):
    # Divisão dos dados em treino e teste para este fold
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    
    # Treinamento
    classifier.fit(X_train, y_train)
    
    # Predição
    preds = classifier.predict(X_test)
    
    # Armazenando para as métricas globais
    todas_predicoes.extend(preds)
    todos_reais.extend(y_test)
    acuracias.append(accuracy_score(y_test, preds))

print(f"Acurácia média (5 folds): {np.mean(acuracias):.4f}")
cm = confusion_matrix(todos_reais, todas_predicoes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Setosa (-1)', 'Versicolour (1)'])
disp.plot(cmap=plt.cm.Blues)
plt.title("Matriz de Confusão (K-Fold k=5)")
plt.show()